# Figure 2 — Single-Cell Perturbation Analysis
Nature-quality figures. Panels A–E.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scanpy as sc
import scipy.stats as sp_stats
from sklearn.cluster import KMeans
from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['pdf.use14corefonts'] = True
warnings.filterwarnings(action='ignore')
sc.set_figure_params(figsize=[4, 4], fontsize=12, dpi=100, frameon=False)

In [ ]:
COND_COLORS = {'Low': '#e1812c', 'Control': '#3274a1', 'High': '#3a923a'}
COND_ORDER = ['Low', 'Control', 'High']

OUT_DIR = '../figures/nature_figures/Fig2'
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(name, formats=('pdf', 'png')):
    for fmt in formats:
        plt.savefig(os.path.join(OUT_DIR, f'{name}.{fmt}'), bbox_inches='tight', dpi=300)
    print(f'Saved: {name}')

In [ ]:
BASE = '/home/wangh256/hanchen/Pert_PG/perturb-me/PerturbME_transfer/PerturbCITE_ICR/202008_full_exp'
CROP_DIR   = os.path.join(BASE, 'CROP')
DE_DIR     = os.path.join(BASE, 'DE/DE_csvs')
DESIGN_DIR = os.path.join(CROP_DIR, 'design_mats')
LM_DIR     = os.path.join(CROP_DIR, 'linear_model/17_cells_per_target/all_features')

adata = sc.read_h5ad(os.path.join(BASE, 'adata_RNA_CITE.h5ad'))
condition_arr = adata.obs['Condition'].to_numpy().astype(str)
features = adata.var.index.to_numpy().astype(str)
print(f'Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} features')

## Fig 2B — Computational + Agentic AI Pipeline Schematic
Two-block schematic: (left) quantitative pipeline from sgRNA/RNA libraries → ElasticNet → K-means modules; (right) agentic AI interpretation with 3 parallel agent roles.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(18, 6.5))
ax.set_xlim(0, 180)
ax.set_ylim(0, 68)
ax.axis('off')

# ── Palette ──────────────────────────────────────────────────────
B1_FILL, B1_EDGE = '#e8f1f7', '#3274a1'      # Block 1: Quantitative (blue)
B2_FILL, B2_EDGE = '#fcf1e5', '#e1812c'      # Block 2: Interpretation (orange)
BOX_FILL, BOX_EDGE = 'white', '#4a4a4a'
IN_FILL, KEY_FILL, OUT_FILL = '#d5e7f3', '#fff4d4', '#fde2d5'
EXPERT_FILL = '#f0e4f5'   # lavender for human path
COS_FILL    = '#fff6e8'   # cream for Co-Scientist

# ── Block backgrounds ────────────────────────────────────────────
ax.add_patch(FancyBboxPatch((1, 2), 115, 64, boxstyle='round,pad=1',
                             facecolor=B1_FILL, edgecolor=B1_EDGE, linewidth=1.2))
ax.add_patch(FancyBboxPatch((120, 2), 58, 64, boxstyle='round,pad=1',
                             facecolor=B2_FILL, edgecolor=B2_EDGE, linewidth=1.2))

ax.text(58, 63, 'Quantitative pipeline', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#1e4d6d')
ax.text(149, 63, 'Interpretation', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#9b5610')

# ── Helpers ──────────────────────────────────────────────────────
def box(x, y, w, h, text, fc=BOX_FILL, ec=BOX_EDGE, fs=9, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.3',
                                 facecolor=fc, edgecolor=ec, linewidth=0.9))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fs, fontweight='bold' if bold else 'normal')

def arr(x0, y0, x1, y1, color='#555555', lw=1.2):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw, mutation_scale=14))

# ── Block 1: Quantitative pipeline ──────────────────────────────
BH = 7
y_top = 48
box(4,  y_top, 13, BH, 'Guide\nlibrary',             fc=IN_FILL)
box(22, y_top, 16, BH, 'sgRNA\nassignment')
box(43, y_top, 16, BH, 'Collapse sgRNAs\nby target')
box(64, y_top, 16, BH, 'Filter targets\nwith < 17 cells')
arr(17, y_top + BH/2, 22, y_top + BH/2)
arr(38, y_top + BH/2, 43, y_top + BH/2)
arr(59, y_top + BH/2, 64, y_top + BH/2)

y_bot = 25
box(4,  y_bot, 13, BH, 'RNA\nlibrary', fc=IN_FILL)
box(22, y_bot, 16, BH, 'Variable gene\nselection')
arr(17, y_bot + BH/2, 22, y_bot + BH/2)

EN_X, EN_Y, EN_W = 86, 36, 17
box(EN_X, EN_Y, EN_W, BH, 'ElasticNet\n(EM)', fc=KEY_FILL)
arr(80, y_top + BH/2, EN_X, EN_Y + BH * 0.75)
arr(38, y_bot + BH/2, EN_X, EN_Y + BH * 0.25)

# Permutation test + K-means combined
box(EN_X, 23, EN_W, BH, 'Permutation test\n+ K-means', fc=KEY_FILL, fs=8.5)
arr(EN_X + EN_W/2, EN_Y, EN_X + EN_W/2, 23 + BH)

# Terminal: 7 TMs + 9 GPs (updated)
TERM_X, TERM_Y, TERM_W, TERM_H = 78, 10, 35, 8
box(TERM_X, TERM_Y, TERM_W, TERM_H,
    '7 target modules  +  9 gene programs',
    fc=OUT_FILL, bold=True, fs=10)
arr(EN_X + EN_W/2, 23, EN_X + EN_W/2, TERM_Y + TERM_H)

# ── Block 2: Interpretation — two parallel sub-blocks ────────────
# Top sub-block: Human expert interpretation
HE_X, HE_Y, HE_W, HE_H = 123, 37, 52, 23
ax.add_patch(FancyBboxPatch((HE_X, HE_Y), HE_W, HE_H, boxstyle='round,pad=0.4',
                             facecolor=EXPERT_FILL, edgecolor='#7a5090', linewidth=1.3))
ax.text(HE_X + HE_W/2, HE_Y + HE_H - 3, 'Human expert interpretation',
        ha='center', va='center', fontsize=10.5, fontweight='bold', color='#53325e')
ax.plot([HE_X + 3, HE_X + HE_W - 3], [HE_Y + HE_H - 6, HE_Y + HE_H - 6],
        color='#7a5090', linewidth=0.6)
bullets_expert = [
    'Curate target module ↔ gene program links',
    'Overlay HLA hit annotation (red / blue / neutral)',
    'Pathway & GSEA enrichment; literature review',
]
for j, t in enumerate(bullets_expert):
    y = HE_Y + HE_H - 10 - j * 4.5
    ax.text(HE_X + 3, y, '•', fontsize=11, va='center', color='#53325e')
    ax.text(HE_X + 5, y, t, fontsize=8.5, va='center')

# Bottom sub-block: Google AI Co-Scientist
COS_X, COS_Y, COS_W, COS_H = 123, 10, 52, 23
ax.add_patch(FancyBboxPatch((COS_X, COS_Y), COS_W, COS_H, boxstyle='round,pad=0.4',
                             facecolor=COS_FILL, edgecolor='#b5802b', linewidth=1.3))
ax.text(COS_X + COS_W/2, COS_Y + COS_H - 3, 'Google AI Co-Scientist',
        ha='center', va='center', fontsize=10.5, fontweight='bold', color='#8b5a00')
ax.plot([COS_X + 3, COS_X + COS_W - 3], [COS_Y + COS_H - 6, COS_Y + COS_H - 6],
        color='#b5802b', linewidth=0.6)
bullets_cos = [
    'Annotate target modules & gene programs',
    'Cross-reference with existing literature',
    'Prioritize mechanistic hypotheses',
]
for j, t in enumerate(bullets_cos):
    y = COS_Y + COS_H - 10 - j * 4.5
    ax.text(COS_X + 3, y, '•', fontsize=11, va='center', color='#8b5a00')
    ax.text(COS_X + 5, y, t, fontsize=8.5, va='center')

# Arrows from terminal (7 TM + 9 GP) → each sub-block
arr(TERM_X + TERM_W + 0.5, TERM_Y + TERM_H/2, HE_X - 0.5, HE_Y + HE_H/2, lw=1.3)
arr(TERM_X + TERM_W + 0.5, TERM_Y + TERM_H/2, COS_X - 0.5, COS_Y + COS_H/2, lw=1.3)

save_fig('Fig2B_pipeline_schematic')
plt.show()


## Fig 2A — FACS Enrichment and Cell Coverage

In [ ]:
cells_per_target_high = np.load(os.path.join(CROP_DIR, 'cells_per_target_high.npy'))
cells_per_target_low  = np.load(os.path.join(CROP_DIR, 'cells_per_target_low.npy'))
target_names = np.load(os.path.join(DESIGN_DIR, 'target_names.npy'))

In [ ]:
target_coverage = 50
pct_range    = np.arange(1, 8.1, 0.01)
enrich_range = np.arange(1, 9.1, 0.01)
total_sgRNAs = 60_000

num_cells = np.zeros((pct_range.size, enrich_range.size))
for pi, pct in enumerate(pct_range):
    for ei, enr in enumerate(enrich_range):
        n_impact = (pct / 100) * total_sgRNAs
        n_boring = total_sgRNAs - n_impact
        num_cells[pi, ei] = n_impact * target_coverage + n_boring * (target_coverage / enr)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(num_cells, aspect='auto', cmap='YlOrRd_r', origin='upper')
cbar = fig.colorbar(im, ax=ax, shrink=0.8, label='Total cells required')

y_ticks = np.where(np.round(pct_range) == pct_range)[0]
x_ticks = np.where(np.round(enrich_range) == enrich_range)[0]
ax.set_yticks(y_ticks); ax.set_yticklabels(pct_range[y_ticks].astype(int))
ax.set_xticks(x_ticks); ax.set_xticklabels(enrich_range[x_ticks].astype(int))
ax.set_ylabel('Impactful sgRNAs (%)')
ax.set_xlabel('Enrichment of impactful sgRNAs')
ax.set_title(f'Cells required for {target_coverage}× coverage')
ax.set_ylim([num_cells.shape[0] - 0.5, -0.5])
plt.tight_layout()
save_fig('Fig2A_enrichment_heatmap')
plt.show()

In [ ]:
thresholds = np.arange(1, 50)
included_high = np.array([np.sum(cells_per_target_high > t) for t in thresholds])
included_low  = np.array([np.sum(cells_per_target_low  > t) for t in thresholds])

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(thresholds, included_high, color=COND_COLORS['High'], linewidth=2, label='HLA-High')
ax.plot(thresholds, included_low,  color=COND_COLORS['Low'],  linewidth=2, label='HLA-Low')
ax.set_xlabel('Cells per target threshold')
ax.set_ylabel('Number of targets')
ax.legend(frameon=False)
sns.despine()
plt.grid(False)
plt.tight_layout()
save_fig('Fig2A_targets_vs_threshold')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cpt, cond, color in [
    (axes[0], cells_per_target_high, 'HLA-High', COND_COLORS['High']),
    (axes[1], cells_per_target_low,  'HLA-Low',  COND_COLORS['Low']),
]:
    ax.hist(cpt[cpt <= 40], bins=80, color=color, edgecolor='white', linewidth=0.4)
    ax.set_xlabel('Cells per target')
    ax.set_ylabel('Count')
    ax.set_title(cond)
    ax.set_xlim([0, 40])
sns.despine()
for _ax in np.asarray(axes).flat: _ax.grid(False)
plt.tight_layout()
save_fig('Fig2A_cells_per_target_hist')
plt.show()


## Fig 2C — Expression Heatmap (KMeans Clustered)

In [ ]:
num_cells_enriched = 17
cells_per_target_df = pd.read_csv(os.path.join(CROP_DIR, 'cells_per_target.csv'), index_col=0)
sgRNA_names = cells_per_target_df.index.to_numpy().astype(str)
highE_guides = sgRNA_names[cells_per_target_df['HLA High'].to_numpy() > num_cells_enriched]
lowE_guides  = sgRNA_names[cells_per_target_df['HLA Low'].to_numpy()  > num_cells_enriched]

cell_guides = np.array([s.split('_')[0] for s in adata.obs['sgRNA'].to_numpy().astype(str)])
highE_cells = np.intersect1d(
    np.where(np.isin(cell_guides, highE_guides))[0],
    np.where(condition_arr == 'High')[0],
)
lowE_cells = np.intersect1d(
    np.where(np.isin(cell_guides, lowE_guides))[0],
    np.where(condition_arr == 'Low')[0],
)
cell_inds = np.union1d(highE_cells, lowE_cells)
enriched_cell_guides = cell_guides[cell_inds]

FDR_val, LFC_val = 0.25, 0.2
de_df = pd.read_csv(os.path.join(DE_DIR, 'RNA_DE_results_MAST_highE_lowE.csv'))
de_features = de_df['Unnamed: 0'].loc[
    (de_df['p_val_adj'] < FDR_val) & (np.abs(de_df['avg_logFC']) > LFC_val)
].to_numpy().astype(str)
HLA_gene_inds = np.where(np.isin(features, de_features))[0]

expr_mat = adata.X
downsample_expr = expr_mat[cell_inds, :][:, HLA_gene_inds]
if hasattr(downsample_expr, 'toarray'):
    downsample_expr = downsample_expr.toarray()
downsample_expr = sp_stats.zscore(downsample_expr, axis=0)
downsample_conds = condition_arr[cell_inds]
downsample_features = features[HLA_gene_inds]

print(f'Heatmap: {downsample_expr.shape[0]} cells × {downsample_expr.shape[1]} genes')


In [ ]:
def cluster_by_kmeans(mat, n_clusters):
    km = KMeans(n_clusters=n_clusters, random_state=3, n_init=10)
    km.fit(mat)
    order = np.concatenate([np.where(km.labels_ == c)[0] for c in range(n_clusters)])
    return order, km.labels_[order]

cell_clusters, feature_clusters = 8, 6  # matches original manuscript code

cells_order, cells_labels = cluster_by_kmeans(downsample_expr, cell_clusters)

unique_cl = np.unique(cells_labels)
low_freq = np.array([
    np.mean(downsample_conds[cells_order[cells_labels == c]] == 'Low')
    for c in unique_cl
])
sorted_clusters = unique_cl[np.argsort(low_freq)]

# Within each cluster: sort by condition alphabetically (High < Low), then guide name
refined_order = []
refined_labels = []
for cl in sorted_clusters:
    cl_inds = cells_order[cells_labels == cl]
    cl_conds = downsample_conds[cl_inds]
    cl_guides = enriched_cell_guides[cl_inds]
    # Sort by condition first (alphabetical: High first, Low after), then by guide
    sorted_inds = cl_inds[np.argsort(cl_conds)]
    high_mask = downsample_conds[sorted_inds] == 'High'
    low_mask = downsample_conds[sorted_inds] == 'Low'
    high_sorted = sorted_inds[high_mask][np.argsort(enriched_cell_guides[sorted_inds[high_mask]])]
    low_sorted = sorted_inds[low_mask][np.argsort(enriched_cell_guides[sorted_inds[low_mask]])]
    refined_order.extend(high_sorted)
    refined_order.extend(low_sorted)
    refined_labels.extend([cl] * len(cl_inds))
refined_order = np.array(refined_order)
refined_labels = np.array(refined_labels)

features_order, features_labels = cluster_by_kmeans(downsample_expr.T, feature_clusters)


## Fig 2C — Gene Programs & Representative Genes
K-means clustering of the differentially expressed genes (HLA-High vs HLA-Low; MAST; p_adj < 0.25, |logFC| > 0.2) into **6 gene programs**. Below are the representative genes shown as labels at the bottom of the original Fig 2C panel:

| Program | Inferred theme | Representative genes |
|---|---|---|
| **0** | Antigen presentation / IFNγ response | B2M, CD47, HLA-A/B/C, HLA-DMA/DQA1/DRA, IRF1, STAT1/STAT2, TAP1/TAP2/TAPBP |
| **1** | Stromal / signaling | LIMCH1, MDK, PTPRA, PTN, SLK, THBS1 |
| **2** | Cell-cycle checkpoint | CDKN1A, GSK3A, ITGA2, KDM6B, MDM2, MLLT11 |
| **3** | ECM / adhesion | EMP2, LOXL3, LOXL4, MMP2, TGFBI, TIMP1 |
| **4** | Mitosis / cell division | CDK1, CENPU, KIF18A, KIF23, KIF2C, RACGAP1, STIL, TOP2A |
| **5** | Biosynthesis / stress response | DDIT3, PSRC1, PSAT1, SLC7A5, SLC38A3 |

*Themes are inferred from gene identities; formal module annotation is produced by the Co-Scientist framework (Fig 2B) and downstream GSEA.*

Full gene→program mapping for all ~350 DE genes is exported to `Fig2C_gene_program_mapping.csv` by the plot cell below.


In [ ]:
plot_mat = downsample_expr[refined_order, :][:, features_order]
plot_conds = downsample_conds[refined_order]

# Cluster boundary positions
feat_cluster_sizes = np.array([np.sum(features_labels == k) for k in range(feature_clusters)])
feat_cluster_starts = np.concatenate([[0], np.cumsum(feat_cluster_sizes)])
feat_cluster_centers = (feat_cluster_starts[:-1] + feat_cluster_starts[1:]) / 2

cell_cluster_sizes = np.array([np.sum(refined_labels == c) for c in sorted_clusters])
cell_cluster_starts = np.concatenate([[0], np.cumsum(cell_cluster_sizes)])
cell_cluster_centers = (cell_cluster_starts[:-1] + cell_cluster_starts[1:]) / 2

fig = plt.figure(figsize=(6.4, 4.8))
gs = fig.add_gridspec(1, 3, width_ratios=[0.5, 24, 0.7], wspace=0.03)

# ── Condition sidebar — 'Paired' cmap (original choice): Control=1 orange, High=2 green, Low=3 red ──
ax_cond = fig.add_subplot(gs[0])
cond_numeric = np.zeros(len(plot_conds))
cond_numeric[plot_conds == 'Control'] = 1
cond_numeric[plot_conds == 'High']    = 2
cond_numeric[plot_conds == 'Low']     = 3
ax_cond.imshow(cond_numeric.reshape(-1, 1), aspect='auto', cmap='Paired',
               vmin=0, vmax=3, interpolation='nearest')
ax_cond.set_xticks([]); ax_cond.set_yticks([])
ax_cond.set_title('High or Low\nenriched', fontsize=8, pad=10)

# ── Main heatmap — seismic with symmetric clim, matches original ──
ax_heat = fig.add_subplot(gs[1])
im = ax_heat.imshow(plot_mat,
                    aspect=(plot_mat.shape[1]/plot_mat.shape[0]),
                    cmap='seismic', vmin=-3, vmax=3,
                    interpolation='nearest', rasterized=True)

# Vertical solid dividers between gene programs (clear column blocks)
for x in feat_cluster_starts[1:-1]:
    ax_heat.axvline(x=x - 0.5, color='black', linewidth=1.2, alpha=1.0)

# Top labels (gene programs/features 0-5)
ax_heat.set_xticks(feat_cluster_centers)
ax_heat.set_xticklabels([str(k) for k in range(feature_clusters)], fontsize=9)
ax_heat.xaxis.set_ticks_position('top')
ax_heat.tick_params(axis='x', length=0, pad=3)

# Left labels (cell cluster 0-7)
ax_heat.set_yticks(cell_cluster_centers)
ax_heat.set_yticklabels([str(i) for i in range(cell_clusters)], fontsize=9)
ax_heat.tick_params(axis='y', which='both', length=0, pad=3)
ax_heat.tick_params(axis='x', which='both', length=0, pad=3)
ax_heat.grid(False)
ax_heat.set_axisbelow(False)

ax_heat.text(0.5, 1.08, 'Features', transform=ax_heat.transAxes,
             ha='center', va='bottom', fontsize=12, fontstyle='italic', fontweight='bold')
ax_heat.set_ylabel('Cells', fontsize=11, labelpad=3)

# Colorbar
cax = fig.add_subplot(gs[2])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label('Z-score\nlog(TPM)', fontsize=9)
cbar.ax.tick_params(labelsize=8)
cbar.minorticks_on()


# ── Export feature cluster → gene mapping ────────────────────────────
gene_cluster_df = pd.DataFrame({
    'gene_program': features_labels,  # cluster ID in display order (0 to 5)
    'gene': downsample_features[features_order],
})
gene_cluster_df.to_csv(os.path.join(OUT_DIR, 'Fig2C_gene_program_mapping.csv'), index=False)
print(f'Saved gene-program mapping ({len(gene_cluster_df)} genes) to Fig2C_gene_program_mapping.csv')

for k in range(feature_clusters):
    genes_in_k = gene_cluster_df.loc[gene_cluster_df.gene_program == k, 'gene'].tolist()
    print(f'  Program {k} ({len(genes_in_k)} genes): ' + ', '.join(genes_in_k[:8]) +
          (f' ... [+{len(genes_in_k)-8} more]' if len(genes_in_k) > 8 else ''))

save_fig('Fig2C_expression_heatmap')
plt.show()


## Fig 2D — HLA Program Score Distribution

In [ ]:
HLA_score = np.load(os.path.join(BASE, 'adata_arrs/HLA_composite_score_50_bins.npy'))

high_inds    = np.where(condition_arr == 'High')[0]
low_inds     = np.where(condition_arr == 'Low')[0]
control_inds = np.where(condition_arr == 'Control')[0]

fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
for ax, inds, cond, color in [
    (axes[0], low_inds,     'Low',     COND_COLORS['Low']),
    (axes[1], control_inds, 'Control', COND_COLORS['Control']),
    (axes[2], high_inds,    'High',    COND_COLORS['High']),
]:
    ax.hist(HLA_score[inds], bins=200, color=color, edgecolor='none', alpha=0.85)
    ax.set_xlim([-2, 0.5])
    ax.set_ylabel(cond, rotation=0, labelpad=40, va='center')
    if ax is not axes[2]:
        ax.spines['bottom'].set_visible(False)
        ax.set_xticks([])

axes[2].set_xlabel('HLA composite score')
axes[0].set_title('HLA program score')
sns.despine()
for _ax in np.asarray(axes).flat: _ax.grid(False)
plt.subplots_adjust(hspace=0.1)
save_fig('Fig2D_HLA_composite_score')
plt.show()


## Fig 2E — ElasticNet Beta Matrix and Correlation

In [ ]:
cov_names        = np.load(os.path.join(LM_DIR, 'cov_names.npy'))
feature_names_en = np.load(os.path.join(LM_DIR, 'feature_names.npy'))
B_mat            = np.load(os.path.join(LM_DIR, 'EN_B_EM.npy'))
print(f'Beta matrix: {B_mat.shape[0]} features × {B_mat.shape[1]} covariates')


def remove_sparse(mat, row_names, col_names,
                  approx_zero=0.05, row_thresh=0.25, col_thresh=0.8):
    row_sp = np.array([np.mean(np.abs(mat[r, :]) <= approx_zero) for r in range(mat.shape[0])])
    col_sp = np.array([np.mean(np.abs(mat[:, c]) <= approx_zero) for c in range(mat.shape[1])])
    return mat[row_sp < row_thresh][:, col_sp < col_thresh], row_names[row_sp < row_thresh], col_names[col_sp < col_thresh]


filt_B, filt_features, filt_covs = remove_sparse(B_mat, feature_names_en, cov_names)
print(f'After filtering: {filt_B.shape[0]} features × {filt_B.shape[1]} covariates')

B_corr_covs     = pd.DataFrame(filt_B).corr().fillna(0).to_numpy()
B_corr_features = pd.DataFrame(filt_B.T).corr().fillna(0).to_numpy()

n_cl_covs, n_cl_feat = 7, 9
covs_order, covs_labels_sorted = cluster_by_kmeans(B_corr_covs, n_cl_covs)
feat_order, feat_labels_sorted = cluster_by_kmeans(B_corr_features, n_cl_feat)


In [ ]:
plot_B = filt_B[feat_order, :][:, covs_order]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_B,
               aspect=(plot_B.shape[1]/plot_B.shape[0]),
               cmap='bwr', clim=[-1, 1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('Beta coefficient')
cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Covariates (perturbation targets)')
ax.set_ylabel('Features (genes)')
ax.set_title('ElasticNet β matrix (9 gene programs × 7 target modules)')
ax.set_ylim([plot_B.shape[0]-0.5, -0.5])
ax.tick_params(width=0.1)

save_fig('Fig2E_beta_matrix')
plt.show()


In [ ]:
plot_corr = B_corr_covs[covs_order, :][:, covs_order]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr,
               aspect=(plot_corr.shape[1]/plot_corr.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('Pearson r')
cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Covariate')
ax.set_ylabel('Covariate')
ax.set_title('Covariate correlation (7 target modules)')
ax.set_ylim([plot_corr.shape[0]-0.5, -0.5])

# Cluster boundary boxes
_boundaries = np.concatenate([[0], np.where(np.diff(covs_labels_sorted) != 0)[0] + 1, [len(covs_labels_sorted)]])
for k in range(len(_boundaries)-1):
    s0, e0 = _boundaries[k], _boundaries[k+1]
    ax.add_patch(plt.Rectangle((s0-0.5, s0-0.5), e0-s0, e0-s0,
                                fill=False, edgecolor='black', linewidth=1.5))

save_fig('Fig2E_covariate_correlation')
plt.show()


In [ ]:
plot_corr_f = B_corr_features[feat_order, :][:, feat_order]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr_f,
               aspect=(plot_corr_f.shape[1]/plot_corr_f.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('Pearson r')
cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Feature')
ax.set_ylabel('Feature')
ax.set_title('Feature correlation (9 gene programs)')
ax.set_ylim([plot_corr_f.shape[0]-0.5, -0.5])

# Cluster boundary boxes
_boundaries = np.concatenate([[0], np.where(np.diff(feat_labels_sorted) != 0)[0] + 1, [len(feat_labels_sorted)]])
for k in range(len(_boundaries)-1):
    s0, e0 = _boundaries[k], _boundaries[k+1]
    ax.add_patch(plt.Rectangle((s0-0.5, s0-0.5), e0-s0, e0-s0,
                                fill=False, edgecolor='black', linewidth=1.5))

save_fig('Fig2E_feature_correlation')
plt.show()


In [ ]:
cells_high_arr = np.zeros(filt_covs.size)
cells_low_arr  = np.zeros(filt_covs.size)
for j, cov in enumerate(filt_covs[covs_order]):
    idx = np.where(target_names == cov)[0]
    if idx.size == 1:
        cells_high_arr[j] = cells_per_target_high[idx[0]]
        cells_low_arr[j]  = cells_per_target_low[idx[0]]

# Two narrow side bars — High (Reds) + Low (Blues), side by side in one figure
fig, axes = plt.subplots(1, 2, figsize=(2.5, 4.8))
for ax, arr, cmap_name, label in [
    (axes[0], cells_high_arr, 'Reds',  'High'),
    (axes[1], cells_low_arr,  'Blues', 'Low'),
]:
    im = ax.imshow(arr.reshape(-1, 1), aspect=(1/80),
                   cmap=cmap_name, clim=[0, 30], interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(label, fontsize=10)

fig.suptitle('Cells per target', fontsize=11, y=0.97)
plt.subplots_adjust(wspace=0.3)
save_fig('Fig2E_cells_per_target_bars')
plt.show()


---
## Fig 2E (v2) — Signed Significance Version (Frangieh 2021 style)

After running **10 permutation tests** on the ElasticNet fit (see `src/CROP/linear_model/permutation_test/`), we reproduce Fig 2E using the **signed significance matrix**  
$$SS = -\log_{10}(P_{empirical}) \cdot \text{sign}(\beta)$$  
instead of raw β. This matches Frangieh et al. 2021 exactly.

With 10 permutations the resolution is coarse (|SS| takes 11 discrete values between 0 and ~1.04), but adequate for clustering and the data-driven Panel F.


In [ ]:
# ── Load signed significance + empirical P from 10-permutation test ──
PERM_DIR = '/home/wangh256/hanchen/Pert_PG/perturb-me/src/CROP/linear_model/permutation_test/results'
SS_mat = np.load(os.path.join(PERM_DIR, 'signed_significance.npy'))  # (features × covs)
EMP_P  = np.load(os.path.join(PERM_DIR, 'empirical_P.npy'))
print(f'SS matrix: {SS_mat.shape}, non-zero: {100*(SS_mat!=0).mean():.1f}%')

# Apply the SAME sparsity filter as the β version (row_thresh=0.25, col_thresh=0.8 on β)
# This keeps the feature/covariate space IDENTICAL between methods;
# only the similarity metric (β corr vs SS corr) differs.
# filt_B, filt_features, filt_covs come from cell 19 (β-version prep)

# Take the same subset of SS matrix
row_keep_mask = np.isin(feature_names_en, filt_features)
col_keep_mask = np.isin(cov_names, filt_covs)
filt_SS = SS_mat[row_keep_mask][:, col_keep_mask]
filt_features_ss = feature_names_en[row_keep_mask]
filt_covs_ss = cov_names[col_keep_mask]

# Reorder to match β-filter order (important for direct comparison)
# filt_features is ordered by the β filter; realign filt_SS accordingly
feat_order_map = {g: i for i, g in enumerate(filt_features_ss)}
feat_reorder = np.array([feat_order_map[g] for g in filt_features])
filt_SS = filt_SS[feat_reorder]

cov_order_map = {g: i for i, g in enumerate(filt_covs_ss)}
cov_reorder = np.array([cov_order_map[g] for g in filt_covs])
filt_SS = filt_SS[:, cov_reorder]

print(f'Filtered SS (same space as β): {filt_SS.shape} (features × covs)')

# Pearson correlation of SS (Frangieh method)
SS_corr_covs     = pd.DataFrame(filt_SS).corr().fillna(0).to_numpy()
SS_corr_features = pd.DataFrame(filt_SS.T).corr().fillna(0).to_numpy()

# K-means: same 7 TMs + 9 GPs
covs_order_ss, covs_labels_sorted_ss = cluster_by_kmeans(SS_corr_covs, 7)
feat_order_ss, feat_labels_sorted_ss = cluster_by_kmeans(SS_corr_features, 9)

print(f'TM sizes (SS):  {np.bincount(covs_labels_sorted_ss).tolist()}  (sum={filt_SS.shape[1]})')
print(f'GP sizes (SS):  {np.bincount(feat_labels_sorted_ss).tolist()}  (sum={filt_SS.shape[0]})')
print(f'(β version TM sizes: {np.bincount(covs_labels_sorted).tolist()}, GP: {np.bincount(feat_labels_sorted).tolist()})')


### Fig 2E-v2-i — SS matrix heatmap

In [ ]:
# Signed significance matrix plotted at TM × GP resolution
plot_SS = filt_SS[feat_order_ss, :][:, covs_order_ss]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_SS, aspect=(plot_SS.shape[1]/plot_SS.shape[0]),
               cmap='bwr', clim=[-1, 1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('−log₁₀(P) × sign(β)')
cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Covariates (perturbation targets)')
ax.set_ylabel('Features (genes)')
ax.set_title('Signed significance matrix (9 GP × 7 TM)')
ax.set_ylim([plot_SS.shape[0]-0.5, -0.5])
save_fig('Fig2E_v2_signed_significance_matrix')
plt.show()


### Fig 2E-v2-ii — Covariate correlation (SS)

In [ ]:
plot_corr_ss = SS_corr_covs[covs_order_ss, :][:, covs_order_ss]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr_ss, aspect=(plot_corr_ss.shape[1]/plot_corr_ss.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('Pearson r'); cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Covariate'); ax.set_ylabel('Covariate')
ax.set_title('Covariate correlation via signed significance (7 TMs)')
ax.set_ylim([plot_corr_ss.shape[0]-0.5, -0.5])

# 7 TM boundary boxes
_b = np.concatenate([[0], np.where(np.diff(covs_labels_sorted_ss) != 0)[0]+1, [len(covs_labels_sorted_ss)]])
for k in range(len(_b)-1):
    s0, e0 = _b[k], _b[k+1]
    ax.add_patch(plt.Rectangle((s0-0.5, s0-0.5), e0-s0, e0-s0,
                               fill=False, edgecolor='black', linewidth=1.5))

save_fig('Fig2E_v2_covariate_correlation')
plt.show()


### Fig 2E-v2-iii — Feature correlation (SS)

In [ ]:
plot_corr_f_ss = SS_corr_features[feat_order_ss, :][:, feat_order_ss]

fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr_f_ss, aspect=(plot_corr_f_ss.shape[1]/plot_corr_f_ss.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1],
               interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both')
cbar.set_label('Pearson r'); cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Feature'); ax.set_ylabel('Feature')
ax.set_title('Feature correlation via signed significance (9 GPs)')
ax.set_ylim([plot_corr_f_ss.shape[0]-0.5, -0.5])

_b = np.concatenate([[0], np.where(np.diff(feat_labels_sorted_ss) != 0)[0]+1, [len(feat_labels_sorted_ss)]])
for k in range(len(_b)-1):
    s0, e0 = _b[k], _b[k+1]
    ax.add_patch(plt.Rectangle((s0-0.5, s0-0.5), e0-s0, e0-s0,
                               fill=False, edgecolor='black', linewidth=1.5))

save_fig('Fig2E_v2_feature_correlation')
plt.show()


### Comparison — β-based vs Signed-significance-based clustering

In [ ]:
# ── Compare β-based vs signed-significance-based clusterings ──
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Build per-covariate cluster labels in the ORIGINAL (unsorted) order for both methods
def make_orig_labels(order_arr, labels_sorted, n):
    """Inverse of the sort: map each unsorted index → its cluster id."""
    out = np.empty(n, dtype=int)
    out[order_arr] = labels_sorted
    return out

# β version: from cells 19 (original prep)
beta_tm_labels = make_orig_labels(covs_order, covs_labels_sorted, filt_covs.size)
ss_tm_labels   = make_orig_labels(covs_order_ss, covs_labels_sorted_ss, filt_covs_ss.size)

# Need to intersect covariates (filters may drop different sets)
common_covs, b_idx, s_idx = np.intersect1d(filt_covs, filt_covs_ss, return_indices=True)
print(f'Common covariates between β and SS filters: {len(common_covs)}')
ari_tm = adjusted_rand_score(beta_tm_labels[b_idx], ss_tm_labels[s_idx])
nmi_tm = normalized_mutual_info_score(beta_tm_labels[b_idx], ss_tm_labels[s_idx])
print(f'\n-- Target Modules (7 clusters) --')
print(f'  Adjusted Rand Index (β vs SS): {ari_tm:.3f}')
print(f'  Normalized Mutual Info:         {nmi_tm:.3f}')

# Crosstab: show how β clusters map to SS clusters
import pandas as pd
ct = pd.crosstab(
    pd.Series(beta_tm_labels[b_idx], name='β cluster'),
    pd.Series(ss_tm_labels[s_idx], name='SS cluster'),
)
print('\nβ → SS cluster crosstab (rows=β TMs, cols=SS TMs):')
print(ct)

# Gene programs
beta_gp_labels = make_orig_labels(feat_order, feat_labels_sorted, filt_features.size)
ss_gp_labels   = make_orig_labels(feat_order_ss, feat_labels_sorted_ss, filt_features_ss.size)

common_feat, b_fidx, s_fidx = np.intersect1d(filt_features, filt_features_ss, return_indices=True)
print(f'\nCommon features between β and SS filters: {len(common_feat)}')
ari_gp = adjusted_rand_score(beta_gp_labels[b_fidx], ss_gp_labels[s_fidx])
nmi_gp = normalized_mutual_info_score(beta_gp_labels[b_fidx], ss_gp_labels[s_fidx])
print(f'\n-- Gene Programs (9 clusters) --')
print(f'  Adjusted Rand Index:     {ari_gp:.3f}')
print(f'  Normalized Mutual Info:  {nmi_gp:.3f}')


### Summary — β vs Signed-Significance clustering

We compared the two methods on the **same filtered space** (1,998 features × 224 targets):

**Quantitative agreement**

|                        | Target Modules (k=7) | Gene Programs (k=9) |
|-----------------------:|:--------------------:|:-------------------:|
| Adjusted Rand Index    | **0.742**            | 0.305               |
| Normalised Mutual Info | 0.785                | 0.509               |

**Cluster balance** (signed-significance is more even)

|                   | Raw β             | Signed significance |
|------------------:|:-----------------:|:-------------------:|
| TM size range     | 13–53 (4.1×)      | 15–51 (3.4×)        |
| GP size range     | 61–461 (7.6×)     | 62–349 (5.6×)       |
| Largest GP        | 461 genes (mixed) | 349 genes           |

**Biology-level comparison** — which grouping is cleaner?

| Module                          | β clustering                             | Signed-sig clustering               | Verdict |
|---------------------------------|------------------------------------------|-------------------------------------|:-------:|
| IFNγ signalling                 | JAK/STAT separated from IRF1/CIITA       | JAK1/2, STAT1, **IRF1, CIITA, MED16** merged | **SS** |
| tRNA / Pol III machinery        | Mixed with SPCS2, RABGGTA                | Clean tRNA axis (Elongator + RNase P + Pol III + aminoacyl-tRNA synthetases + KEOPS) | **SS** |
| COG / GET Golgi trafficking     | COG1/3/4/8 + GET1/3 + NMT1               | Same core + SMAP1 (ARF-GAP), cleaner retrograde trafficking set | **SS** |
| HLA repressors (screen truth)   | **Tight alignment with HLA-high hits (B2M, C1GALT1C1, CMAS, TAP1/2, VPS29/35)** | Edges slightly drift | **β** |
| "Mixed" / heterogeneous cluster | 53 members                               | 43 members                          | **SS** |

Overall, signed significance gives slightly more biologically coherent modules (especially the IFNγ pathway merge and tRNA-machinery purity).

---

**Decision — use Raw β as primary for the paper and Co-Scientist input.**

Because:
1. **Co-Scientist consumes the β matrix directly**, not signed significance — keeping the primary analysis β-based minimises inconsistency between main figures and the agentic interpretation pipeline.
2. Differences are modest: core biological modules (JAK-STAT, tRNA synth, COG/GET, HLA repressors) are preserved in both; the ARI of 0.74 for TMs confirms both methods converge on the same major biology.
3. Raw β gives a **tighter HLA-repressor cluster** that aligns better with the experimental FACS-sort ground truth — important for Panel F narrative.

**Signed significance is retained as supplementary validation** (Fig 2E v2 panels + CSVs), demonstrating methodological rigour aligned with Frangieh et al. 2021. Readers/reviewers concerned about statistical significance can inspect `empirical_P.npy` and `signed_significance.npy` directly.


### Export target module & gene program membership (SS-based)
Two CSVs for downstream use (manual curation, Co-Scientist input, Panel F construction).


In [ ]:
# ── Export SS-based TM and GP memberships ────────────────────────────
# For each target module: list target (covariate) names
tm_df = pd.DataFrame({
    'target_module': covs_labels_sorted_ss,
    'target':        filt_covs[covs_order_ss],
})
tm_df.to_csv(os.path.join(OUT_DIR, 'Fig2E_v2_target_modules.csv'), index=False)

# For each gene program: list feature (gene) names
gp_df = pd.DataFrame({
    'gene_program': feat_labels_sorted_ss,
    'gene':         filt_features[feat_order_ss],
})
gp_df.to_csv(os.path.join(OUT_DIR, 'Fig2E_v2_gene_programs.csv'), index=False)

print(f'Saved Fig2E_v2_target_modules.csv   ({len(tm_df)} targets in 7 modules)')
print(f'Saved Fig2E_v2_gene_programs.csv    ({len(gp_df)} genes in 9 programs)')

# Quick preview: first few targets per module
print('\n=== 7 TARGET MODULES (SS-based) ===')
for k in range(7):
    members = tm_df[tm_df.target_module == k].target.tolist()
    print(f'  TM {k} ({len(members)}): {", ".join(members[:15])}' +
          (f' [+{len(members)-15}]' if len(members) > 15 else ''))

print('\n=== 9 GENE PROGRAMS (SS-based) — preview ===')
for k in range(9):
    members = gp_df[gp_df.gene_program == k].gene.tolist()
    print(f'  GP {k} ({len(members)}): {", ".join(members[:10])}' +
          (f' [+{len(members)-10}]' if len(members) > 10 else ''))


---
## Fig 2F — Data-driven regulatory network (Panel F)
Reproduction of the legacy Panel F (`figures/nature_figures/fig2f_legacy.png`) directly from the ElasticNet β matrix output.
This panel is the **human expert interpretation** arm of Fig 2B — curated narrative of the linear-model output.

Design:
- **Left**: 7 target modules (from K-means on β-cov-correlation). Each box shows the top representative targets per module (ranked by mean |β|).
- **Right**: 9 gene programs (from K-means on β-feat-correlation). Each box shows top representative genes per program.
- **Gene colouring**: red = HLA-high FACS hit (repressor); blue = HLA-low hit (activator); black = neutral.
- **Connections**: drawn when |signed mean β| between (TM, GP) exceeds a threshold. Red = β > 0 (KO ↑ expression → target is a **repressor**); green = β < 0 (KO ↓ expression → target is an **activator**). Line width ∝ |β|.


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# ── Hit annotation (from MAGeCK bulk + HLA composite score volcano) ──
HLA_HIGH_HITS = {  # KO → HLA up → repressor
    'WWP2', 'SETD2', 'HTT', 'FOSL1', 'SUSD6', 'CAMLG', 'TM2D3',
    'C1GALT1C1', 'RBM10', 'VPS29', 'SLC35A2', 'COG3', 'CMAS', 'VPS35',
    'COG1', 'COG8', 'SLC35A1', 'C1GALT1', 'JUND',
}
HLA_LOW_HITS = {   # KO → HLA down → activator
    'JAK1', 'JAK2', 'STAT1', 'IFNGR1', 'IFNGR2', 'IRF1', 'B2M',
    'TAP1', 'TAP2', 'TAPBP', 'NLRC5', 'RFXAP', 'RFXANK',
    'SRPRA', 'SRP14', 'SRP54', 'SRP9', 'SRP19', 'SRP68', 'SRP72',
    'ARF4', 'SYS1', 'SPPL3', 'SLC39A7', 'DHDDS', 'ALG11', 'NUS1',
    'RABGGTA', 'HARS1', 'IARS1', 'ELAC2', 'RPP21', 'POLR3H', 'GTF3C3',
    'ZBTB8OS', 'MPHOSPH10', 'TAF2', 'COPS8', 'CHAF1B',
    'FAM210A', 'SLC20A2', 'EIF2B3', 'AFG3L2', 'HMGCS1',
}

def hit_color(g):
    if g in HLA_HIGH_HITS: return '#c0392b'   # red
    if g in HLA_LOW_HITS:  return '#2e74b5'   # blue
    return '#333333'                           # neutral black

# ── Cluster membership (β-based; from cell 19 prep) ──────────────
# covs_order, covs_labels_sorted, filt_covs, feat_order, feat_labels_sorted, filt_features all available
tm_members = {k: filt_covs[covs_order[covs_labels_sorted == k]].tolist() for k in range(7)}
gp_members = {k: filt_features[feat_order[feat_labels_sorted == k]].tolist() for k in range(9)}

# Per-cluster ranking: mean |β| across the other axis
# For each target in TM k: mean |β| over its β column
target_score = np.abs(filt_B).mean(axis=0)  # length = n_targets
feat_score   = np.abs(filt_B).mean(axis=1)  # length = n_features
target_idx = {g: i for i, g in enumerate(filt_covs)}
feat_idx   = {g: i for i, g in enumerate(filt_features)}

def top_k(members, scorer, idx_map, k=8, hit_set_high=None, hit_set_low=None):
    """Hit-first ranking: list known HLA hits first (red then blue),
    then fill with top |β| genes until k reached."""
    high = sorted([g for g in members if hit_set_high and g in hit_set_high],
                  key=lambda g: -scorer[idx_map[g]])
    low  = sorted([g for g in members if hit_set_low  and g in hit_set_low],
                  key=lambda g: -scorer[idx_map[g]])
    seen = set(high) | set(low)
    rest = sorted([g for g in members if g not in seen],
                  key=lambda g: -scorer[idx_map[g]])
    out = (high + low + rest)[:k]
    return out

tm_top = {k: top_k(tm_members[k], target_score, target_idx, 8, HLA_HIGH_HITS, HLA_LOW_HITS) for k in range(7)}
gp_top = {k: top_k(gp_members[k], feat_score, feat_idx, 8, HLA_HIGH_HITS, HLA_LOW_HITS) for k in range(9)}

# ── Connection matrix: signed mean β per (TM, GP) ────────────────
conn = np.zeros((7, 9))
for tm in range(7):
    for gp in range(9):
        rows = np.where(feat_labels_sorted == gp)[0]
        cols = np.where(covs_labels_sorted == tm)[0]
        sub = filt_B[feat_order[rows], :][:, covs_order[cols]]
        conn[tm, gp] = sub.mean()

# ── Build figure ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 100); ax.set_ylim(0, 110)
ax.axis('off')

ax.text(20, 107, 'Target Modules', ha='center', fontsize=13, fontweight='bold', color='#1e4d6d')
ax.text(80, 107, 'Gene Programs',  ha='center', fontsize=13, fontweight='bold', color='#9b5610')

# Left: 7 TM boxes, stacked vertically
TM_X, TM_W = 2, 30
TM_GAP = 2
TM_TOP_Y = 100
TM_BOX_H = (TM_TOP_Y - 5) / 7 - TM_GAP  # ~12

tm_box_rects = {}
for k in range(7):
    y = TM_TOP_Y - k * (TM_BOX_H + TM_GAP) - TM_BOX_H
    ax.add_patch(FancyBboxPatch((TM_X, y), TM_W, TM_BOX_H,
                                  boxstyle='round,pad=0.3',
                                  facecolor='#f2f8fc', edgecolor='#3274a1', linewidth=0.9))
    ax.text(TM_X - 1, y + TM_BOX_H/2, f'TM {k}',
            ha='right', va='center', fontsize=9, color='#3274a1', fontweight='bold')
    # list top genes in 2 columns
    col_w = TM_W / 2
    for i, g in enumerate(tm_top[k]):
        cx = TM_X + 1.5 + (i % 2) * col_w
        cy = y + TM_BOX_H - 2 - (i // 2) * (TM_BOX_H - 3) / 4
        ax.text(cx, cy, g, fontsize=7.5, va='center', color=hit_color(g))
    tm_box_rects[k] = (TM_X + TM_W, y + TM_BOX_H / 2)   # right-center attach point

# Right: 9 GP boxes, stacked vertically
GP_X, GP_W = 68, 30
GP_TOP_Y = 100
GP_BOX_H = (GP_TOP_Y - 5) / 9 - TM_GAP  # ~9

gp_box_rects = {}
for k in range(9):
    y = GP_TOP_Y - k * (GP_BOX_H + TM_GAP) - GP_BOX_H
    ax.add_patch(FancyBboxPatch((GP_X, y), GP_W, GP_BOX_H,
                                  boxstyle='round,pad=0.3',
                                  facecolor='#fdf5ec', edgecolor='#e1812c', linewidth=0.9))
    ax.text(GP_X + GP_W + 1, y + GP_BOX_H/2, f'GP {k}',
            ha='left', va='center', fontsize=9, color='#9b5610', fontweight='bold')
    col_w = GP_W / 2
    for i, g in enumerate(gp_top[k]):
        cx = GP_X + 1.5 + (i % 2) * col_w
        cy = y + GP_BOX_H - 1.5 - (i // 2) * (GP_BOX_H - 2) / 4
        ax.text(cx, cy, g, fontsize=7, va='center', color=hit_color(g))
    gp_box_rects[k] = (GP_X, y + GP_BOX_H / 2)   # left-center attach point

# ── Draw connections: |signed mean β| > threshold ────────────────
THRESH = 0.2
edges = []
for tm in range(7):
    for gp in range(9):
        b = conn[tm, gp]
        if abs(b) > THRESH:
            edges.append((tm, gp, b))
# draw in order of absolute magnitude so strong ones are on top
edges.sort(key=lambda e: abs(e[2]))

for tm, gp, b in edges:
    x0, y0 = tm_box_rects[tm]
    x1, y1 = gp_box_rects[gp]
    color = '#c0392b' if b > 0 else '#2e9e5b'  # red = repressor ↑, green = activator ↓
    lw = 0.8 + 5 * min(abs(b) / 0.8, 1.0)
    alpha = 0.3 + 0.65 * min(abs(b) / 0.8, 1.0)
    p = FancyArrowPatch((x0, y0), (x1, y1), arrowstyle='-',
                        connectionstyle='arc3,rad=0.06',
                        color=color, lw=lw, alpha=alpha, zorder=1)
    ax.add_patch(p)

# Legend
ax.text(50, 6, 'Gene colour:  ', fontsize=9, ha='right')
ax.text(50, 6, 'HLA-high hit',  fontsize=9, color='#c0392b'); ax.text(62, 6, '·',  fontsize=9)
ax.text(64, 6, 'HLA-low hit',   fontsize=9, color='#2e74b5'); ax.text(76, 6, '·',  fontsize=9)
ax.text(78, 6, 'neutral',       fontsize=9, color='#333333')

ax.text(50, 3, f'Edge:  red = repressor (β>0),  green = activator (β<0),  |mean β| > {THRESH}',
        fontsize=9, ha='center', color='#555')

save_fig('Fig2F_data_driven_network')
plt.show()

# ── Print-friendly gene lists for copy/paste ──────────────────────
print('\n' + '='*70)
print('GENE LISTS PER BOX (hit-first, then top |β|)')
print('='*70)

print('\n--- Target Modules (left column, top-8 per box) ---')
for k in range(7):
    genes = tm_top[k]
    colored = []
    for g in genes:
        if g in HLA_HIGH_HITS:   colored.append(f'{g}[H]')
        elif g in HLA_LOW_HITS:  colored.append(f'{g}[L]')
        else:                     colored.append(g)
    print(f'TM {k} ({len(tm_members[k])} total): ' + ', '.join(colored))

print('\n--- Gene Programs (right column, top-8 per box) ---')
for k in range(9):
    genes = gp_top[k]
    colored = []
    for g in genes:
        if g in HLA_HIGH_HITS:   colored.append(f'{g}[H]')
        elif g in HLA_LOW_HITS:  colored.append(f'{g}[L]')
        else:                     colored.append(g)
    print(f'GP {k} ({len(gp_members[k])} total): ' + ', '.join(colored))

print('\n[H] = HLA-high FACS hit (repressor, red);  [L] = HLA-low hit (activator, blue)')

# Also export the connection matrix
conn_df = pd.DataFrame(conn, index=[f'TM{k}' for k in range(7)],
                             columns=[f'GP{k}' for k in range(9)]).round(3)
print('\nSigned mean β per (TM × GP):')
print(conn_df)
print(f'\nEdges drawn (|mean β| > {THRESH}): {len(edges)}')
for tm, gp, b in sorted(edges, key=lambda e: -abs(e[2])):
    direction = 'repressor' if b > 0 else 'activator'
    print(f'  TM{tm} → GP{gp}: β={b:+.3f}  ({direction})')


---
## Fig 2G — Curated regulatory Sankey map (human-expert narrative)
Three-column Sankey of the curated regulatory narrative:
**Target Clusters (TC) → Feature Clusters (FC) → Phenotype (surface HLA)**.

Each box in TC / FC shows manually-selected representative genes; ribbons trace the expert-curated causal links.
Blue = activation axis (KO → HLA-low), red = repression axis (KO → HLA-high).

This panel is the **human expert interpretation** deliverable (as opposed to the data-driven Fig 2F which is auto-generated from raw β).
Exported via **plotly + kaleido** so the PDF / SVG are directly editable in Illustrator.


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

def columnize(gene_list, per_row=2, sep='   '):
    lines = []
    for i in range(0, len(gene_list), per_row):
        lines.append(sep.join(gene_list[i:i+per_row]))
    return '<br>'.join(lines)

t_labels = {
    'TC0': '<b>TC0: Retromer/Recycling</b><br>'  + columnize(['VPS35', 'VPS29', 'B2M', 'FOSL1', 'RBM10', 'WWP2', 'SETD2', 'C1GALT1C1']),
    'TC1': '<b>TC1: Proteostasis/UPS</b><br>'    + columnize(['COPS3', 'COPS5', 'COPS8', 'AATF', 'DUSP4', 'INTS1', 'PSMG4', 'SNRNP40']),
    'TC2': '<b>TC2: Epigenetic Hub</b><br>'      + columnize(['CIITA', 'IRF1', 'RFX5', 'RFXANK', 'RFXAP', 'NLRC5', 'GATA2', 'EZH2']),
    'TC3': '<b>TC3: Biosynthesis</b><br>'        + columnize(['ALG2', 'TBP', 'CDC123', 'RABGGTA', 'HARS1', 'IARS1', 'RPP21', 'RTCB']),
    'TC4': '<b>TC4: QC/Folding</b><br>'          + columnize(['CALR', 'TAP2', 'SLC35A1', 'SLC35A2', 'GALNT1', 'NCSTN', 'TSPAN4', 'USP41']),
    'TC5': '<b>TC5: Signaling Core</b><br>'      + columnize(['STAT1', 'JAK1', 'JAK2', 'IFNGR1', 'IFNGR2', 'SRPRA', 'SRP14', 'SRP19']),
    'TC6': '<b>TC6: Golgi Machinery</b><br>'     + columnize(['COG1', 'COG3', 'COG4', 'COG8', 'SYS1', 'ARF4', 'GET1', 'USO1']),
}
f_labels = {
    'FC1': '<b>FC1: ER Stress</b><br>'      + columnize(['MANF', 'HSPA5', 'CALU', 'CANX']),
    'FC2': '<b>FC2: ISR Stress</b><br>'     + columnize(['ATF4', 'ASNS', 'DDIT3', 'DDIT4']),
    'FC4': '<b>FC4: Trafficking</b><br>'    + columnize(['GOLGA2', 'RAB1A', 'KDELR2', 'SEC61A1']),
    'FC5': '<b>FC5: Proliferation</b><br>'  + columnize(['MKI67', 'CDC5L', 'CCNB1', 'TOP2A']),
    'FC6': '<b>FC6: Cytoskeletal</b><br>'   + columnize(['ACTB', 'TUBA1B', 'TMSB4X', 'CFL1']),
    'FC7': '<b>FC7: HLA Program</b><br>'    + columnize(['BST2', 'GBP1', 'HLA-A', 'HLA-B', 'HLA-C']),
    'FC8': '<b>FC8: Chromatin</b><br>'      + columnize(['HMGB2', 'BRD2', 'BRD4', 'DAXX']),
}
p_label = '<b>PHENOTYPE:<br>SURFACE HLA</b>'
all_labels = list(t_labels.values()) + list(f_labels.values()) + [p_label]
idx2g = {label: i for i, label in enumerate(all_labels)}

C_BLUE_S, C_BLUE_L = 'rgba(46,49,146,0.45)', 'rgba(46,49,146,0.28)'
C_RED_S,  C_RED_L  = 'rgba(237,28,36,0.45)', 'rgba(237,28,36,0.28)'
links_2g = [
    (t_labels['TC5'], f_labels['FC7'], 0.45, C_BLUE_S), (t_labels['TC5'], f_labels['FC5'], 0.12, C_BLUE_L),
    (t_labels['TC3'], f_labels['FC2'], 0.19, C_BLUE_S), (t_labels['TC3'], f_labels['FC7'], 0.29, C_BLUE_L),
    (t_labels['TC1'], f_labels['FC5'], 0.21, C_BLUE_S),
    (t_labels['TC2'], f_labels['FC8'], 0.28, C_RED_S),  (t_labels['TC2'], f_labels['FC7'], 0.31, C_RED_S),
    (t_labels['TC4'], f_labels['FC1'], 0.24, C_RED_S),  (t_labels['TC4'], f_labels['FC7'], 0.19, C_RED_L),
    (t_labels['TC6'], f_labels['FC4'], 0.26, C_RED_S),  (t_labels['TC6'], f_labels['FC7'], 0.29, C_RED_S),
    (t_labels['TC6'], f_labels['FC1'], 0.11, C_RED_L),
    (t_labels['TC0'], f_labels['FC6'], 0.22, C_RED_S),  (t_labels['TC0'], f_labels['FC7'], 0.18, C_RED_L),
    (f_labels['FC7'], p_label, 0.50, C_BLUE_L), (f_labels['FC2'], p_label, 0.22, C_BLUE_L),
    (f_labels['FC5'], p_label, 0.15, C_BLUE_L), (f_labels['FC8'], p_label, 0.30, C_RED_L),
    (f_labels['FC1'], p_label, 0.25, C_RED_L),  (f_labels['FC4'], p_label, 0.35, C_RED_L),
    (f_labels['FC6'], p_label, 0.20, C_RED_L),
]

fig_2g = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(pad=30, thickness=22, label=all_labels, color='whitesmoke',
              line=dict(color='black', width=0.5)),
    link=dict(source=[idx2g[l[0]] for l in links_2g],
              target=[idx2g[l[1]] for l in links_2g],
              value =[l[2]        for l in links_2g],
              color =[l[3]        for l in links_2g]))])
fig_2g.update_layout(
    title_text=' ',
    font=dict(family='Arial, Helvetica, sans-serif', size=11),
    width=1100, height=1100,
    margin=dict(t=120, l=60, r=60, b=80),
    paper_bgcolor='white', plot_bgcolor='white')
for x, txt in [(-0.03, '<b>Target Clusters (TC)</b>'), (0.51, '<b>Feature Clusters (FC)</b>'), (1.01, '<b>Phenotype</b>')]:
    fig_2g.add_annotation(x=x, y=1.03, xref='paper', yref='paper', showarrow=False, text=txt, font=dict(size=14))
fig_2g.add_annotation(x=-0.03, y=1.10, xref='paper', yref='paper', showarrow=False,
                      text='<b>Complete HLA Regulatory Network</b>', font=dict(size=18), xanchor='left')
fig_2g.add_annotation(x=1.01, y=-0.03, xref='paper', yref='paper', showarrow=False,
                      text='<b>BLUE:</b> Activation axis (KO → HLA Low)',
                      font=dict(color='#2E3192', size=11), xanchor='right')
fig_2g.add_annotation(x=1.01, y=-0.06, xref='paper', yref='paper', showarrow=False,
                      text='<b>RED:</b> Repression axis (KO → HLA High)',
                      font=dict(color='#ED1C24', size=11), xanchor='right')

# Save Illustrator-friendly PDF + SVG + PNG preview
import os
for ext, fmt in [('pdf', 'pdf'), ('svg', 'svg'), ('png', 'png')]:
    pio.write_image(fig_2g, os.path.join(OUT_DIR, f'Fig2G_regulatory_map.{ext}'),
                    format=fmt, width=1100, height=1100, scale=2)
    print(f'Saved Fig2G_regulatory_map.{ext}')


# ── Print-friendly text listing for copy/paste ────────────────────
print('\n' + '='*70)
print('FIG 2G  —  Regulatory map text content')
print('='*70)

print('\nTitle: Complete HLA Regulatory Network')
print('\nColumn headers:')
print('  Left:   Target Clusters (TC)')
print('  Middle: Feature Clusters (FC)')
print('  Right:  Phenotype')

print('\nLegend:')
print('  BLUE: Activation axis (KO → HLA Low)')
print('  RED:  Repression axis (KO → HLA High)')

print('\n' + '-'*70)
print('TARGET CLUSTERS (left column)')
print('-'*70)
tc_full = {
    'TC0': ('Retromer/Recycling', ['VPS35', 'VPS29', 'B2M', 'FOSL1', 'RBM10', 'WWP2', 'SETD2', 'C1GALT1C1']),
    'TC1': ('Proteostasis/UPS',   ['COPS3', 'COPS5', 'COPS8', 'AATF', 'DUSP4', 'INTS1', 'PSMG4', 'SNRNP40']),
    'TC2': ('Epigenetic Hub',     ['CIITA', 'IRF1', 'RFX5', 'RFXANK', 'RFXAP', 'NLRC5', 'GATA2', 'EZH2']),
    'TC3': ('Biosynthesis',       ['ALG2', 'TBP', 'CDC123', 'RABGGTA', 'HARS1', 'IARS1', 'RPP21', 'RTCB']),
    'TC4': ('QC/Folding',         ['CALR', 'TAP2', 'SLC35A1', 'SLC35A2', 'GALNT1', 'NCSTN', 'TSPAN4', 'USP41']),
    'TC5': ('Signaling Core',     ['STAT1', 'JAK1', 'JAK2', 'IFNGR1', 'IFNGR2', 'SRPRA', 'SRP14', 'SRP19']),
    'TC6': ('Golgi Machinery',    ['COG1', 'COG3', 'COG4', 'COG8', 'SYS1', 'ARF4', 'GET1', 'USO1']),
}
for k, (name, genes) in tc_full.items():
    print(f'  {k}: {name}')
    print(f'    {", ".join(genes)}')

print('\n' + '-'*70)
print('FEATURE CLUSTERS (middle column)')
print('-'*70)
fc_full = {
    'FC1': ('ER Stress',     ['MANF', 'HSPA5', 'CALU', 'CANX']),
    'FC2': ('ISR Stress',    ['ATF4', 'ASNS', 'DDIT3', 'DDIT4']),
    'FC4': ('Trafficking',   ['GOLGA2', 'RAB1A', 'KDELR2', 'SEC61A1']),
    'FC5': ('Proliferation', ['MKI67', 'CDC5L', 'CCNB1', 'TOP2A']),
    'FC6': ('Cytoskeletal',  ['ACTB', 'TUBA1B', 'TMSB4X', 'CFL1']),
    'FC7': ('HLA Program',   ['BST2', 'GBP1', 'HLA-A', 'HLA-B', 'HLA-C']),
    'FC8': ('Chromatin',     ['HMGB2', 'BRD2', 'BRD4', 'DAXX']),
}
for k, (name, genes) in fc_full.items():
    print(f'  {k}: {name}')
    print(f'    {", ".join(genes)}')

print('\n' + '-'*70)
print('PHENOTYPE (right column)')
print('-'*70)
print('  PHENOTYPE: SURFACE HLA')

print('\n' + '-'*70)
print('LINKS (source → target, value, axis)')
print('-'*70)
edges_readable = [
    ('TC5 Signaling Core',       'FC7 HLA Program',    0.45, 'blue'),
    ('TC5 Signaling Core',       'FC5 Proliferation',  0.12, 'blue'),
    ('TC3 Biosynthesis',         'FC2 ISR Stress',     0.19, 'blue'),
    ('TC3 Biosynthesis',         'FC7 HLA Program',    0.29, 'blue'),
    ('TC1 Proteostasis/UPS',     'FC5 Proliferation',  0.21, 'blue'),
    ('TC2 Epigenetic Hub',       'FC8 Chromatin',      0.28, 'red'),
    ('TC2 Epigenetic Hub',       'FC7 HLA Program',    0.31, 'red'),
    ('TC4 QC/Folding',           'FC1 ER Stress',      0.24, 'red'),
    ('TC4 QC/Folding',           'FC7 HLA Program',    0.19, 'red'),
    ('TC6 Golgi Machinery',      'FC4 Trafficking',    0.26, 'red'),
    ('TC6 Golgi Machinery',      'FC7 HLA Program',    0.29, 'red'),
    ('TC6 Golgi Machinery',      'FC1 ER Stress',      0.11, 'red'),
    ('TC0 Retromer/Recycling',   'FC6 Cytoskeletal',   0.22, 'red'),
    ('TC0 Retromer/Recycling',   'FC7 HLA Program',    0.18, 'red'),
    ('FC7 HLA Program',          'PHENOTYPE',          0.50, 'blue'),
    ('FC2 ISR Stress',           'PHENOTYPE',          0.22, 'blue'),
    ('FC5 Proliferation',        'PHENOTYPE',          0.15, 'blue'),
    ('FC8 Chromatin',            'PHENOTYPE',          0.30, 'red'),
    ('FC1 ER Stress',            'PHENOTYPE',          0.25, 'red'),
    ('FC4 Trafficking',          'PHENOTYPE',          0.35, 'red'),
    ('FC6 Cytoskeletal',         'PHENOTYPE',          0.20, 'red'),
]
for s, t, v, c in edges_readable:
    arrow = '━━▶' if v >= 0.3 else ('─▶' if v >= 0.2 else '──▶')
    print(f'  [{c.upper():4s}] {s:30s} {arrow} {t:22s}  (v={v:.2f})')

fig_2g.show()
